# **PubMedQA**



## **Problem Statement**

Build a **hybrid search & retrieval system** over biomedical research abstracts using the [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) dataset.

### Architecture

```
                        ┌─── Dense (BGE-Large) ──────┐
Query ──► Embed Query ──┤                            ├──► RRF Fusion ──► Final Ranked Results         
                        └─── Sparse (SPLADE) ────────┘
```

### Key Concepts for this notebook

- **Dense retrieval + Sparse retrieval**
- **Hybrid search**
- **RRF**



In [1]:
# Installs
!pip install -q qdrant-client fastembed datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 6.6 MB/s eta 0:00:00


In [ ]:
import json

import numpy as np
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    SparseVector,
    PointStruct,
    QueryRequest,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
    ScoredPoint,
)
from transformers import AutoTokenizer

import fastembed
from fastembed import SparseEmbedding, SparseTextEmbedding, TextEmbedding

print(f"FastEmbed version: {fastembed.__version__}")

In [3]:
# Load & Explore the PubMedQA Dataset
# Load the PubMedQA artificial split
dataset = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

In [4]:
print(f"Total instances: {len(dataset)}")
print(f"Features: {dataset.column_names}")

Total instances: 211269
Features: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


In [5]:
# Work with a subset of 100 for this assignment
dataset = dataset.select(range(100))
print(f"Working with {len(dataset)} instances")

Working with 100 instances


In [6]:
# Convert to DataFrame
df = dataset.to_pandas()

In [7]:
df.head()

,pubid,question,context,long_answer,final_decision
0,25429730,Are group 2 innate lymphoid cells ( ILC2s ) in...,{'contexts': ['Chronic rhinosinusitis (CRS) is...,"As ILC2s are elevated in patients with CRSwNP,...",yes
1,25433161,Does vagus nerve contribute to the development...,{'contexts': ['Phosphatidylethanolamine N-meth...,Neuronal signals via the hepatic vagus nerve c...,yes
2,25445714,Does psammaplin A induce Sirtuin 1-dependent a...,{'contexts': ['Psammaplin A (PsA) is a natural...,PsA significantly inhibited MCF-7/adr cells pr...,yes
3,25431941,Is methylation of the FGFR2 gene associated wi...,{'contexts': ['This study examined links betwe...,We identified a novel biologically plausible c...,yes
4,25432519,Do tumor-infiltrating immune cell profiles and...,{'contexts': ['Tumor microenvironment immunity...,Breast cancer immune cell subpopulation profil...,yes


In [8]:
# Data Cleaning & Document Construction
# Check for duplicate pubids
print(f"Total rows:           {len(df)}")
print(f"Unique pubids:        {df['pubid'].nunique()}")
print(f"Duplicate pubids:     {df.duplicated(subset=['pubid']).sum()}")

print(f"\n--- Missing Values ---")
print(df.isnull().sum())

print(f"\n--- Empty Strings ---")
print(f"Empty questions:      {(df['question'] == '').sum()}")
print(f"Empty long_answers:   {(df['long_answer'] == '').sum()}")

print(f"\n--- Final Decision Distribution ---")
print(df['final_decision'].value_counts())

Total rows:           100
Unique pubids:        100
Duplicate pubids:     0

--- Missing Values ---
pubid             0
question          0
context           0
long_answer       0
final_decision    0
dtype: int64

--- Empty Strings ---
Empty questions:      0
Empty long_answers:   0

--- Final Decision Distribution ---
final_decision
yes    89
no     11
Name: count, dtype: int64


In [9]:
# Construct document Text
def build_document_text(row) -> str:
    parts = []
    context = row["context"]
    if isinstance(context, str):
        context = json.loads(context)
    for para, label in zip(context["contexts"], context["labels"]):
        parts.append(f"[{label}] {para}")
    parts.append(f"[CONCLUSION] {row['long_answer']}")
    mesh_terms = context.get("meshes", [])
    if len(mesh_terms) > 0:
        parts.append(f"[MeSH Terms] {', '.join(mesh_terms)}")
    return "\n\n".join(parts)

df["combined_text"] = df.apply(build_document_text, axis=1)

print(f"Total documents to embed: {len(df)}")
print(f"\nSample document (first 500 chars):")
print(df["combined_text"].iloc[0][:500] + "...")

Total documents to embed: 100

Sample document (first 500 chars):
[BACKGROUND] Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.

[OBJECTIVE] The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of diseas...


In [ ]:
# Generating Embeddings
# Load models
sparse_model_name = "prithvida/Splade_PP_en_v1"
dense_model_name = "BAAI/bge-large-en-v1.5"

sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)
dense_model = TextEmbedding(model_name=dense_model_name, batch_size=32)

print("✅ Both models loaded!")

/tmp/ipykernel_12471/3437358668.py:4: DeprecationWarning: The right spelling is prithivida/Splade_PP_en_v1. Support of this name will be removed soon, please fix the model_name
  sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# Helper functions
def make_sparse_embedding(texts: list[str]) -> list[SparseEmbedding]:
    return list(sparse_model.embed(texts, batch_size=32))

def make_dense_embedding(texts: list[str]):
    return list(dense_model.embed(texts))

In [ ]:
# Embed full catalog
product_texts = df["combined_text"].tolist()
print(f"Embedding {len(product_texts)} documents...")

In [ ]:
%%time
print("⏳ Generating sparse embeddings (SPLADE)...")
df["sparse_embedding"] = make_sparse_embedding(product_texts)
print("✅ Done!")

In [ ]:
%%time
print("⏳ Generating dense embeddings (BGE-Large)...")
df["dense_embedding"] = make_dense_embedding(product_texts)
print("✅ Done!")

In [ ]:
# Qdrant indexing
# Create collection

client = QdrantClient(":memory:")

collection_name = "pubmedqa"

client.create_collection(
    collection_name,
    vectors_config={
        "text-dense": VectorParams(
            size=1024,
            distance=Distance.COSINE,
        )
    },
    sparse_vectors_config={
        "text-sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,
            )
        )
    },
)

print(f"✅ Collection '{collection_name}' created with dense + sparse vector support")

In [ ]:
# Prepare & Upload Points
def make_points(df: pd.DataFrame) -> list[PointStruct]:
    """Convert DataFrame rows into Qdrant PointStruct objects."""
    sparse_vectors = df["sparse_embedding"].tolist()
    doc_texts = df["combined_text"].tolist()
    dense_vectors = df["dense_embedding"].tolist()
    rows = df.to_dict(orient="records")

    points = []
    for idx, (text, sparse_vector, dense_vector) in enumerate(
        zip(doc_texts, sparse_vectors, dense_vectors)
    ):
        sparse_vector = SparseVector(
            indices=sparse_vector.indices.tolist(),
            values=sparse_vector.values.tolist()
        )

        point = PointStruct(
            id=idx,
            payload={
                "text": text,
                "pubid": int(rows[idx]["pubid"]),
                "question": rows[idx]["question"],
                "final_decision": rows[idx]["final_decision"],
            },
            vector={
                "text-sparse": sparse_vector,
                "text-dense": dense_vector.tolist(),
            },
        )
        points.append(point)
    return points

points = make_points(df)
print(f"Prepared {len(points)} points for indexing")

In [ ]:
# Upload to Qdrant
client.upsert(collection_name, points)
print(f"✅ {len(points)} points indexed in Qdrant!")

In [ ]:
# Hybrid Search
# Search function
def search(query_text: str, top_k: int = 10):
    query_sparse_vectors = make_sparse_embedding([query_text])
    query_dense_vector = make_dense_embedding([query_text])

    search_results = client.query_batch_points(
        collection_name=collection_name,
        requests=[
            QueryRequest(
                query=query_dense_vector[0].tolist(),
                using="text-dense",
                limit=top_k,
                with_payload=True,
            ),
            QueryRequest(
                query=SparseVector(
                    indices=query_sparse_vectors[0].indices.tolist(),
                    values=query_sparse_vectors[0].values.tolist(),
                ),
                using="text-sparse",
                limit=top_k,
                with_payload=True,
            ),
        ],
    )

    return [search_results[0].points, search_results[1].points]

In [ ]:
# Search tests
# Query 1: Exact drug name
query_text = "Does metformin improve insulin sensitivity in type 2 diabetes?"
search_results = search(query_text)

dense_results, sparse_results = search_results[0], search_results[1]

print(f"Query: '{query_text}'")
print(f"\n{'='*70}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(dense_results[:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

print(f"\n{'='*70}")
print(f"SPARSE (Lexical) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(sparse_results[:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

In [ ]:
# Query 2: Conceptual search
query_text = "Can physical activity help with mental health conditions?"
search_results_2 = search(query_text)

print(f"Query: '{query_text}'")
print(f"\n{'='*70}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_2[0][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

print(f"\n{'='*70}")
print(f"SPARSE (Lexical) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_2[1][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

Query: 'Can physical activity help with mental health conditions?'

DENSE (Semantic) Results — Top 5:
  1. [Score: 0.6804] [BACKGROUND] In recent years, the relationship between physical activity (PA) and smoking cessation ...
  2. [Score: 0.6018] [BACKGROUND] Active commuters have lower risk of chronic disease. Understanding which of the, to som...
  3. [Score: 0.5836] [BACKGROUND] Disturbed sleep is associated with mood disorders. Both depression and insomnia may inc...
  4. [Score: 0.5798] [OBJECTIVE] Immunologic abnormalities have been found in bipolar disorder but pentraxin 3, a marker ...
  5. [Score: 0.5724] [OBJECTIVE] To determine whether older adults with type 2 diabetes mellitus and cognitive dysfunctio...

SPARSE (Lexical) Results — Top 5:
  1. [Score: 11.8652] [BACKGROUND] In recent years, the relationship between physical activity (PA) and smoking cessation ...
  2. [Score: 6.6863] [OBJECTIVE] Researchers studying the murine response to stress generally use mice housed un

In [ ]:
# Query 3: Mixed search - exact + conceptual
query_text = "Is statin therapy associated with reduced mortality?"
search_results_3 = search(query_text)

print(f"Query: '{query_text}'")
print(f"\n{'='*70}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_3[0][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

print(f"\n{'='*70}")
print(f"SPARSE (Lexical) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_3[1][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

Query: 'Is statin therapy associated with reduced mortality?'

DENSE (Semantic) Results — Top 5:
  1. [Score: 0.6138] [BACKGROUND] Obese patients with idiopathic pulmonary fibrosis (IPF) have higher 90-day mortality af...
  2. [Score: 0.6004] [BACKGROUND] The aims of this study were to determine the effect of failed prior endovascular treatm...
  3. [Score: 0.5992] [OBJECTIVE] The aim of this study was to determine the association between 25-hydroxyvitamin D (25(O...
  4. [Score: 0.5928] [BACKGROUND] Hyperglycemia on admission is associated with increased mortality rates in patients wit...
  5. [Score: 0.5927] [OBJECTIVE] To review UK newspaper reports relating to Do Not Attempt Cardiopulmonary Resuscitation ...

SPARSE (Lexical) Results — Top 5:
  1. [Score: 11.0263] [BACKGROUND] Hyperglycemia on admission is associated with increased mortality rates in patients wit...
  2. [Score: 9.2990] [BACKGROUND] Studies performed in Andean populations living in high altitude, indicate that the 

In [ ]:
# Reciprocal Rank Fusion (RRF)
# RRF Implementation
def rrf(rank_lists, k=60, default_rank=1000):
    all_items = set(item for rank_list in rank_lists for item, _ in rank_list)
    item_to_index = {item: idx for idx, item in enumerate(all_items)}

    rank_matrix = np.full((len(all_items), len(rank_lists)), default_rank)

    for list_idx, rank_list in enumerate(rank_lists):
        for item, rank in rank_list:
            rank_matrix[item_to_index[item], list_idx] = rank

    rrf_scores = np.sum(1.0 / (k + rank_matrix), axis=1)

    sorted_indices = np.argsort(-rrf_scores)
    index_to_item = {idx: item for item, idx in item_to_index.items()}
    sorted_items = [(index_to_item[idx], rrf_scores[idx]) for idx in sorted_indices]

    return sorted_items

def rank_list(search_result: list[ScoredPoint]):
    """Converts Qdrant search results to (id, rank) pairs"""
    return [(point.id, rank + 1) for rank, point in enumerate(search_result)]

In [ ]:
# Apply RRF to the Search Results
query_text = "Does metformin improve insulin sensitivity in type 2 diabetes?"
search_results = search(query_text)

dense_rank_list = rank_list(search_results[0])
sparse_rank_list = rank_list(search_results[1])

rrf_rank_list = rrf([dense_rank_list, sparse_rank_list])

print(f"\n🏆 HYBRID SEARCH RESULTS for: '{query_text}'")
print(f"{'='*70}")

fused_records = client.retrieve(
    collection_name=collection_name,
    ids=[item[0] for item in rrf_rank_list]
)

record_by_id = {r.id: r for r in fused_records}

for rank, (item_id, score) in enumerate(rrf_rank_list, 1):
    record = record_by_id[item_id]
    title = record.payload['text'][:80].replace('\n', ' ')

    in_dense = any(id == item_id for id, _ in dense_rank_list)
    in_sparse = any(id == item_id for id, _ in sparse_rank_list)
    source = "BOTH" if (in_dense and in_sparse) else ("Dense only" if in_dense else "Sparse only")

    print(f"  {rank:>2}. [{source:<12}] {title}...")


🏆 HYBRID SEARCH RESULTS for: 'Does metformin improve insulin sensitivity in type 2 diabetes?'
   1. [BOTH        ] [OBJECTIVE] To determine whether older adults with type 2 diabetes mellitus and ...
   2. [BOTH        ] [BACKGROUND] Recently, the role of osteoprotegerin (OPG) in the pathogenesis of ...
   3. [BOTH        ] [BACKGROUND] Plasma levels of pancreatic polypeptide (PP) rise upon food intake....
   4. [BOTH        ] [BACKGROUND] Hyperglycemia on admission is associated with increased mortality r...
   5. [BOTH        ] [BACKGROUND] Guillotine below-knee amputation (BKA) for wet gangrene is an unfor...
   6. [Sparse only ] [BACKGROUND] Stroke is the third leading cause of invalidism and death in indust...
   7. [Dense only  ] [OBJECTIVE] The aim of this study was to determine the association between 25-hy...
   8. [Sparse only ] [BACKGROUND] The role of Kdr (VEGFR-2/Flk-1) in vascular formation has been well...
   9. [Dense only  ] [BACKGROUND] Obese patients with idiopathic 